# Numerical check of the degree of $\mathcal{F}_1$

For $(d,n)=(3,4)$ and generic $\mathcal{H}$ the eigenvector variety $\mathcal{E}(\mathcal{H})$ is a quartic surface in $\mathbb{P}^3$ with the $4\times 4$ linear determinantal representation
$$f(x_1,x_2,x_3,x_4) \;=\; \det\bigl[\,H_1\, x \mid H_2\, x \mid H_3\, x \mid x\,\bigr], \qquad H_1,H_2,H_3\in\mathbb{C}^{4\times 4}.$$
This notebook computes the degree of the hypersurface inside $\mathbb{P}^{34}$ swept out by these surfaces as $(H_1,H_2,H_3)$ varies.

By Proposition 2.6 in our paper, the hypersurface is irreducible and its degree is $320{,}112$. This was first obtained via intersection theory in [LLHV, Theorem 2].  We recover the number numerically by intersecting with a random line and counting parameter fibers using `monodromy_solve` from `HomotopyContinuation.jl`.

**Pipeline:**
1. Build the parametrization $(H_1,H_2,H_3)\mapsto P=(\text{coeffs of }f)$ of $\mathcal{F}_1$.
2. Obtain a finite-to-one parametrization $Z$ of the whole locus.  Verify numerically that the Jacobian rank does not drop.
3. Set up an affine-linear cut $c\cdot[1;Z]=0$ of complementary dimension and find one starting fiber.
4. Run `monodromy_solve` to count all preimages of the cut.

**Reference.** M. Leal, C. Lozano Huerta, M. Vite, *The Noether–Lefschetz locus of surfaces in $\mathbb{P}^3$ formed by determinantal surfaces*, arXiv:[2303.09028](https://arxiv.org/abs/2303.09028).

In [2]:
using HomotopyContinuation
using LinearAlgebra

@var x[1:4]  H₁[1:4,1:4]  H₂[1:4,1:4]  H₃[1:4,1:4]

coeffs = vcat(vec(H₁), vec(H₂), vec(H₃));   # 48 entries of (H₁, H₂, H₃)

## Building $f$ and its coefficient vector $P$

Form $f$ and take $P\in\mathbb{C}^{35}$, the coefficient vector of $f$ in the monomial basis of degree-$4$ forms in $4$ variables. It parameterizes the variety $\mathcal{F}_1$. The affine dimension of $\mathcal{F}_1$ is the numerical rank of the Jacobian of $P$. This is a hypersurface so we verify that the rank is $34$.

In [3]:
f = expand(det([H₁*x  H₂*x  H₃*x  x]))
P = coefficients(f, x)

# The affine dimension of F_1 equals the numerical rank of the Jacobian of P
J = differentiate(P, coeffs)
println("affine dim of F_1 = ",
        rank(evaluate.(J, coeffs => randn(length(coeffs)))))

affine dim of F_1 = 34


## Finite-to-one parametrization

Set one row of each of $H_1,H_2,H_3$ to zero and two further entries to $1$, leaving $34$ free parameters.  Let $Z$ be the resulting coefficient vector.  This yields a finite-to-one parametrization of the whole locus as long as the Jacobian rank does not drop. We verify it numerically.

In [ ]:
elim = [H₁[4,:]; H₂[3,:]; H₃[2,:]]   # entries set to 0
os   = [H₁[3,1], H₂[2,1]]             # entries set to 1

Z    = evaluate.(P, [elim; os] => [zeros(length(elim)); ones(length(os))])
Jz   = differentiate(Z, coeffs)
dimF1 = rank(evaluate.(Jz, coeffs => randn(length(coeffs))))

println("affine dim of F_1 (after chart) = ", dimF1)

## Start fiber

A random line in $\mathbb{P}^{34}$ pulls back to an affine system $c\cdot[1;Z]=0$, linear in $c$.  We find a start pair by sampling a $\text{start\_sol}$ and solving for $\text{start\_params}$ by linear algebra.

In [6]:
vars_Z = variables(System(Z))      # remaining 34 entries of (H₁, H₂, H₃)

@var c[1:dimF1, 1:length(Z) + 1]
c_vec = vec(c)

# Parametric line section:  F(x; c) = c · [1; Z(x)]
F = System(c * [1; Z], parameters = c_vec)

# Sample a start point in vars_Z-space and require the cut to vanish there.
# The resulting equations are linear in c_vec, so we solve them by linear algebra.
start_sol = rand(ComplexF64, length(vars_Z))
param_eqs = evaluate(F, start_sol)              # vector of polys in c_vec

M = convert(Matrix{ComplexF64},
            [differentiate(p, q) for p in param_eqs, q in c_vec])
b = evaluate.(param_eqs, c_vec => zeros(length(c_vec)))

# Particular solution of  M·c = -b  perturbed within the null space
N = nullspace(M)
start_params = M \ (-b) + N * rand(ComplexF64, size(N, 2))

println("residual at start fiber: ",
        norm(evaluate.(param_eqs, c_vec => start_params)))

residual at start fiber: 3.236933504104724e-14


## Monodromy

Starting from the start fiber, `monodromy_solve` finds all parameter solutions of the affine cut by tracking loops in $c$-space.  The function $\mathrm{dist}(u,v)=\|G(u)-G(v)\|_\infty$ identifies parameter solutions whose images on the line agree.  The count of distinct images is $\deg\mathcal{F}_1$.  (This computation is slow.  We recommend running it on many threads (>100) and on a server.)

In [ ]:
G    = System(Z, variables = vars_Z)
dist = (u, v) -> norm(G(u) - G(v), Inf)

result = monodromy_solve(F, start_sol, start_params;
                         distance = dist,
                         threading = true,
                         catch_interrupt = true)

## Verify distinct images

`monodromy_solve`'s `dist` does not always merge solutions with the same image.  Recompare by rounding $G(u)$ entrywise to tolerance $10^{-5}$ and counting unique values.  The result is $\deg\mathcal{F}_1$.

In [ ]:
# Two solutions agree on the line iff their images G(u) coincide.
# Compare entry-wise with tolerance 1e-5.
sols = solutions(result)
key(s) = Tuple((round(real(z), digits=5), round(imag(z), digits=5)) for z in G(s))
unique_sols = unique(key, sols)

println("raw solutions: ", length(sols))
println("deg F_1     = ", length(unique_sols))